# ビタビアルゴリズムを実装して最適なラベル列を求める

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/14_sequence_labeling/viterbi.ipynb)

NLP クックブック 14-05 ／ 系列ラベリング

HMM や CRF の復号に使われるビタビアルゴリズムを、自分で実装して動きを確かめたい。

In [1]:
import numpy as np

def viterbi(emission, transition, start):
    """emission: (系列長, ラベル数), transition: (ラベル数, ラベル数), start: (ラベル数,) の対数スコア"""
    T, K = emission.shape
    score = start + emission[0]
    back = np.zeros((T, K), dtype=int)
    for t in range(1, T):
        cand = score[:, None] + transition      # cand[i, j] = 直前が i で今が j のときのスコア
        back[t] = cand.argmax(axis=0)
        score = cand.max(axis=0) + emission[t]
    path = [int(score.argmax())]
    for t in range(T - 1, 0, -1):               # 後ろからたどる
        path.append(int(back[t, path[-1]]))
    return path[::-1], float(score.max())

labels = ["名詞", "動詞", "助詞"]
words = ["猫", "が", "魚", "を", "食べる"]
log = lambda p: np.log(np.array(p) + 1e-12)
start = log([0.7, 0.2, 0.1])
transition = log([[0.2, 0.2, 0.6],     # 名詞 → 名詞 / 動詞 / 助詞
                  [0.3, 0.3, 0.4],     # 動詞 →
                  [0.6, 0.39, 0.01]])  # 助詞 →
emission = log([[0.9, 0.05, 0.05],     # 猫
                [0.3, 0.1, 0.6],       # が (「蛾」かもしれない)
                [0.9, 0.05, 0.05],     # 魚
                [0.1, 0.1, 0.8],       # を
                [0.1, 0.85, 0.05]])    # 食べる

path, score = viterbi(emission, transition, start)
print([f"{w}/{labels[i]}" for w, i in zip(words, path)], round(score, 3))

['猫/名詞', 'が/助詞', '魚/名詞', 'を/助詞', '食べる/動詞'] -3.938


In [2]:
# 貪欲法 (各位置で独立に最大のラベルを選ぶ) との比較、および総当たりとの一致の確認
from itertools import product

greedy = emission.argmax(axis=1).tolist()
def total(p):
    return start[p[0]] + emission[0, p[0]] + sum(transition[a, b] + emission[t, b] for t, (a, b) in enumerate(zip(p, p[1:]), 1))
brute = max(product(range(3), repeat=5), key=total)
print("貪欲法:", [labels[i] for i in greedy], " 総当たり:", [labels[i] for i in brute], " 一致:", list(brute) == path)

貪欲法: ['名詞', '助詞', '名詞', '助詞', '動詞']  総当たり: ['名詞', '助詞', '名詞', '助詞', '動詞']  一致: True


- 計算量は O(系列長 × ラベル数²)。総当たりの O(ラベル数^系列長) を動的計画法で避けている。
- 遷移スコアに「O → I- は −∞」のような制約を入れれば、ニューラルモデルの出力に対しても正しい BIO 系列だけを出力させられる。